In [ ]:
#!/usr/bin/env python3
# mmca_rho_vs_beta.py
"""
MMCA sweep: 阈值实验, kappa 6 9 12, beta 0.0-0.7
 - generate R independent random 3-uniform hypergraphs H(n,m,3)
 - run MMCA on each graph from same initial p_i(0)=init_frac
 - collect final rho (mean p across nodes) per run -> compute mean/std across runs
 - save summary CSV and runs-matrix CSV for each kappa

Outputs (per kappa):
  out_mmca/mmca_kappa{K}_summary.csv   (columns: beta, mean_rho, std_rho)
  out_mmca/mmca_kappa{K}_runs.csv      (rows correspond to beta, cols to runs)
"""
import os, math, time
import numpy as np
import random
import pandas as pd


# -------------------- Hypergraph generator H(n,m,3) --------------------
def generate_hypergraph_H_n_m_3(n, kappa, seed=None, batch_size=3000):
    if seed is not None:
        np.random.seed(seed); random.seed(seed)
    d = 3
    m = int(round(n * kappa / d))
    max_possible = math.comb(n, d)
    if m > max_possible:
        raise ValueError("m > C(n,3)")
    nodes = np.arange(n)
    edges_set = set()
    while len(edges_set) < m:
        tris = np.random.choice(nodes, size=(batch_size, d), replace=True)
        for i in range(tris.shape[0]):
            row = tris[i]
            if len(np.unique(row)) < d:
                row = np.random.choice(nodes, size=(d,), replace=False)
            tris[i] = np.sort(row)
        for row in tris:
            if len(edges_set) >= m:
                break
            edges_set.add(tuple(row.tolist()))
    hyperedges = np.array(list(edges_set), dtype=int)
    hyperedges.sort(axis=1)
    return hyperedges

# -------------------- MMCA implementation on one hypergraph --------------------
class MMCA:
    def __init__(self, hyperedges, n):
        self.hyperedges = np.asarray(hyperedges, dtype=int)
        self.n = n
        self.m = self.hyperedges.shape[0]
        # endpoints for vectorized ops
        self.i = self.hyperedges[:,0]; self.j = self.hyperedges[:,1]; self.k = self.hyperedges[:,2]
        # build node incidence
        self.node_edge_pos = [[] for _ in range(n)]
        for ei in range(self.m):
            a,b,c = self.hyperedges[ei]
            self.node_edge_pos[a].append((ei,0))
            self.node_edge_pos[b].append((ei,1))
            self.node_edge_pos[c].append((ei,2))

    def run(self, beta_d, mu, eta, gamma, init_frac, max_iters=1000, tol=1e-9):
        # p initialization (uniform)
        p = np.full(self.n, init_frac, dtype=float)
        Theta = np.ones(self.m, dtype=float)
        for it in range(max_iters):
            # compute edge-wise prod of others for each position
            p_i = p[self.i]; p_j = p[self.j]; p_k = p[self.k]
            prod_pos0 = p_j * p_k
            prod_pos1 = p_i * p_k
            prod_pos2 = p_i * p_j

            # compute Q_node = product_{e in E_i} (1 - Theta_e * beta * prod_others)
            Q = np.ones(self.n, dtype=float)
            # loop nodes (could be vectorized more, but clarity first)
            for node in range(self.n):
                pv = 1.0
                for (ei,pos) in self.node_edge_pos[node]:
                    if pos == 0:
                        oth = prod_pos0[ei]
                    elif pos == 1:
                        oth = prod_pos1[ei]
                    else:
                        oth = prod_pos2[ei]
                    pv *= (1.0 - Theta[ei] * beta_d * oth)
                Q[node] = pv

            # MMCA update
            p_next = (1.0 - p) * (1.0 - Q) + (1.0 - mu) * p

            # Theta update based on current p (Phi = prod p inside each edge)
            Phi = p_i * p_j * p_k
            Theta = Theta * (1.0 - eta * Phi) + gamma * (1.0 - Theta)
            Theta = np.clip(Theta, 0.0, 1.0)

            delta = np.max(np.abs(p_next - p))
            p = p_next
            if delta < tol:
                break
        # return final mean infection density and iteration count
        return float(p.mean()), int(it+1)
    
# -------------------- Parameters (change if you want) --------------------
N = 1500               # node count
KAPPAS = [6.0, 9.0, 12.0]
R = 50                 # repeats per beta (you requested 50)
# BETAS = np.linspace(0.02, 0.6, 30)   # beta sweep
BETAS = np.concatenate([
    np.arange(0.00, 0.1, 0.02),
    np.arange(0.1, 0.3, 0.02),
    np.arange(0.3, 0.4, 0.02),
    np.arange(0.4, 0.7, 0.02)
])
MU = 0.1
ETA = 0.6
GAMMA = 0.02
INIT_FRAC = 0.3      # initial p_i(0)
MAX_ITERS = 300       # MMCA max iterations
TOL = 1e-5             # convergence tolerance (max |p_next - p|)
SEED_BASE = 12345

OUT_DIR = "out_mmca"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------- outer loop: sweep beta, repeat graphs --------------------
t0 = time.time()
for kappa in KAPPAS:
    print(f"\n=== MMCA: kappa={kappa}  R={R}  betas={len(BETAS)} ===")
    m = int(round(N * kappa / 3))
    runs_matrix = np.zeros((len(BETAS), R), dtype=float)
    iters_matrix = np.zeros((len(BETAS), R), dtype=int)
    for ib, beta in enumerate(BETAS):
        for r in range(R):
            gen_seed = SEED_BASE + r
            hyperedges = generate_hypergraph_H_n_m_3(N, kappa, seed=gen_seed)
            model = MMCA(hyperedges, N)
            rho_final, iters = model.run(beta_d=beta, mu=MU, eta=ETA, gamma=GAMMA,
                                         init_frac=INIT_FRAC, max_iters=MAX_ITERS, tol=TOL)
            runs_matrix[ib, r] = rho_final
            iters_matrix[ib, r] = iters
        mean_rho = runs_matrix[ib].mean()
        std_rho = runs_matrix[ib].std(ddof=0)
        print(f"k={kappa} beta={beta:.3f}  mean_rho={mean_rho:.4f} std={std_rho:.4f}")
    # save files per kappa
    summary_df = pd.DataFrame({
        'beta': BETAS,
        'mean_rho': runs_matrix.mean(axis=1),
        'std_rho': runs_matrix.std(axis=1, ddof=0)
    })
    runs_df = pd.DataFrame(runs_matrix, columns=[f'run{r:03d}' for r in range(R)])
    summary_fn = os.path.join(OUT_DIR, f"mmca_kappa{int(kappa)}_summary.csv")
    runs_fn = os.path.join(OUT_DIR, f"mmca_kappa{int(kappa)}_runs.csv")
    summary_df.to_csv(summary_fn, index=False, float_format="%.4f")
    runs_df.to_csv(runs_fn, index=False, float_format="%.4f")
    print(f"Saved {summary_fn} and {runs_fn}")

print("All done. Total time: {:.1f}s".format(time.time()-t0))



=== MMCA: kappa=6.0  R=50  betas=36 ===
k=6.0 beta=0.000  mean_rho=0.0001 std=0.0000
k=6.0 beta=0.020  mean_rho=0.0001 std=0.0000
k=6.0 beta=0.040  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.060  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.080  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.100  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.120  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.140  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.160  mean_rho=0.0000 std=0.0000
k=6.0 beta=0.180  mean_rho=0.3587 std=0.0038
k=6.0 beta=0.200  mean_rho=0.3787 std=0.0003
k=6.0 beta=0.220  mean_rho=0.4021 std=0.0006
k=6.0 beta=0.240  mean_rho=0.4206 std=0.0006
k=6.0 beta=0.260  mean_rho=0.4372 std=0.0006
k=6.0 beta=0.280  mean_rho=0.4526 std=0.0006
k=6.0 beta=0.300  mean_rho=0.4665 std=0.0007
k=6.0 beta=0.320  mean_rho=0.4793 std=0.0007
k=6.0 beta=0.340  mean_rho=0.4912 std=0.0007
k=6.0 beta=0.360  mean_rho=0.5023 std=0.0007
k=6.0 beta=0.380  mean_rho=0.5127 std=0.0008
k=6.0 beta=0.400  mean_rho=0.5225 std=0.0008
k=6.0 beta=0.4

In [8]:
runs_matrix

array([[8.99072019e-05, 8.99072019e-05, 8.99072019e-05, ...,
        8.99072019e-05, 8.99072019e-05, 8.99072019e-05],
       [5.27004311e-05, 5.25301002e-05, 5.26254665e-05, ...,
        5.27175575e-05, 5.26933207e-05, 4.72977598e-05],
       [4.40469873e-05, 4.28969001e-05, 4.85021342e-05, ...,
        4.92360662e-05, 4.41538115e-05, 4.30488288e-05],
       ...,
       [7.26848299e-01, 7.27228763e-01, 7.27034695e-01, ...,
        7.26648046e-01, 7.26653242e-01, 7.27178399e-01],
       [7.31218386e-01, 7.31597401e-01, 7.31404126e-01, ...,
        7.31016994e-01, 7.31022130e-01, 7.31547095e-01],
       [7.35408329e-01, 7.35785819e-01, 7.35593350e-01, ...,
        7.35205891e-01, 7.35210659e-01, 7.35735584e-01]])